# rust_dataflow

> Fill in a module description here

In [ ]:
# | default_exp rust_dataflow

In [ ]:
# | export
import os
import subprocess
from datetime import datetime
from pathlib import Path

import jinja2
import networkx as nx

from spannerflow.config import Config
from spannerflow.engine import Engine
from spannerflow.graph_utils import (
    create_iter_graph,
    find_ingress_nodes,
    find_output,
    find_sources,
    get_common_cols,
    get_minus_cols,
    get_node_schema,
    reduced_graph,
    traverse_cycle,
)

# Class RustDataflow

In [ ]:
# | export


class RustDataflow:
    DATAFLOW_TO_RUST_TYPES = {
        "DATA_TYPE_STRING": "String",
        "DATA_TYPE_INT": "i32",
        "DATA_TYPE_FLOAT": "OrderedFloat<f32>",
        "DATA_TYPE_BOOL": "bool",
    }

    PYTHON_TO_DATAFLOW_TYPES = {
        str: "DATA_TYPE_STRING",
        int: "DATA_TYPE_INT",
        float: "DATA_TYPE_FLOAT",
        bool: "DATA_TYPE_BOOL",
        object: "DATA_TYPE_STRING",
    }

    def __init__(self, engine: Engine, config: Config):
        self._engine = engine
        self._config = config
        self._query_id = 0
        self._cargo_file_name = "Cargo.toml"
        self._template_loader = jinja2.FileSystemLoader(
            searchpath=self._config.TEMPLATES_PATH
        )
        self._template_env = jinja2.Environment(loader=self._template_loader)

    def get_input_schema_types(self, node: int | str) -> list[str]:
        collections = self._engine.get_collections()
        return [x for x in collections[str(node)]]

    def get_input_schema(self, node: int | str) -> list[str]:
        collections = self._engine.get_collections()
        return [self.DATAFLOW_TO_RUST_TYPES[x] for x in collections[str(node)]]

    def get_sources_data(
        self,
        graph: nx.DiGraph,
        nodes_schema_types_dict: dict[str | int, list[str]],
    ) -> dict[str | int, dict[str, str | int | list[str]]]:
        data = dict()
        for source in find_sources(graph):
            gr_node = graph.nodes[source]
            op = gr_node["op"]
            data[source] = {
                "name": source,
                "op": op,
            }

            if source not in nodes_schema_types_dict:
                data[source]["schema"] = self.get_input_schema(source)
            else:
                data[source]["schema"] = [
                    self.DATAFLOW_TO_RUST_TYPES[t]
                    for t in nodes_schema_types_dict[source]
                ]
                if op == "get_const":
                    data[source]["consts"] = self._engine._serialize_row(
                        nodes_schema_types_dict[source],
                        [gr_node["const_dict"][col] for col in gr_node["schema"]],
                    )
        return data

    @staticmethod
    def get_col_schema(cols: list[str]) -> str:
        if not cols:
            return "0"
        if len(cols) > 1:
            return f"({', '.join(cols)})"
        else:
            return cols[0]

    def get_join_code(
        self,
        graph: nx.DiGraph,
        node: str | int,
        anchor: str | int | None = None,
        in_iterate: bool = False,
        nodes_schema_types_dict: dict[str | int, list[str]] | None = None,
    ) -> str:
        """Returns a string of rust code for joining two collections on the common columns.
        If no columns are common, the join is a cross product.
        """
        join1, join2 = list(graph.pred[node])
        out_node_str = self.get_node_str(node)
        join1_str = self.get_node_str(join1)
        join2_str = self.get_node_str(join2)

        if in_iterate:
            if node == anchor:
                out_node_str = str(anchor)
            if join1 == anchor:
                join1_str = join1
            if join2 == anchor:
                join2_str = join2

        common_cols = get_common_cols(graph, join1, join2)
        common_schema = self.get_col_schema(common_cols)

        join1_uncommon_schema = self.get_col_schema(
            get_minus_cols(graph, join1, common_cols)
        )
        join2_uncommon_schema = self.get_col_schema(
            get_minus_cols(graph, join2, common_cols)
        )
        out_join1_uncommon_schema = (
            join1_uncommon_schema if join1_uncommon_schema != "0" else "_"
        )

        out_join2_uncommon_schema = (
            join2_uncommon_schema if join2_uncommon_schema != "0" else "_"
        )

        return f"""let {out_node_str} = {join1_str}.map(|{get_node_schema(graph, join1)}| ({common_schema}, {join1_uncommon_schema}))
                            .join(&{join2_str}.map(|{get_node_schema(graph, join2)}| ({common_schema}, {join2_uncommon_schema})))
                            .map(|({common_schema if common_schema != "0" else "_"}, ({out_join1_uncommon_schema}, {out_join2_uncommon_schema}))| ({get_node_schema(graph, node)}));"""

    def get_union_code(
        self,
        graph: nx.DiGraph,
        node: str | int,
        anchor: str | int | None = None,
        in_iterate: bool = False,
        nodes_schema_types_dict: dict[str | int, list[str]] | None = None,
    ) -> str:
        """Returns a string of rust code for unioning two collections."""
        preds = [
            pred
            for pred in graph.pred[node]
            if "reduced" not in graph.get_edge_data(pred, node)
        ]

        prev_node1_str = self.get_node_str(
            preds[0], anchor=anchor, in_iterate=in_iterate
        )
        node_str = self.get_node_str(node, anchor=anchor, in_iterate=in_iterate)

        if len(preds) == 1:
            return f"let {' mut' if not in_iterate and node_str == 'node_' + str(node) else ''} {node_str} = {prev_node1_str}.clone();"
        prev_node2_str = self.get_node_str(
            preds[1], anchor=anchor, in_iterate=in_iterate
        )
        return f"let{' mut' if not in_iterate and node_str == 'node_' + str(node) else ''} {node_str} = {prev_node1_str}.concat(&{prev_node2_str});"

    def get_node_str(
        self, node: str | int, anchor: str | int | None = None, in_iterate: bool = False
    ) -> str:
        if in_iterate and node == anchor:
            return str(anchor)
        return f"node_{node}"

    def validate_node(self, graph: nx.DiGraph, node: str | int) -> None:
        """Validates the node in the graph."""
        gr_node = graph.nodes[node]
        preds = [graph.nodes[key] for key in graph.pred[node].keys()]
        match gr_node["op"]:
            case "union":
                preds = [
                    pred
                    for pred in graph.pred[node]
                    if "reduced" not in graph.get_edge_data(pred, node)
                ]
                if len(preds) not in (1, 2):
                    raise ValueError(
                        "Union node has invalid number of predecessors: ",
                        (len(preds), node),
                    )
            case "rename" | "select" | "project" | "groupby":
                if len(preds) != 1:
                    raise ValueError(
                        f"{gr_node['op']} node has invalid number of predecessors: ",
                        (len(preds), node),
                    )
            case "product" | "join":
                if len(preds) != 2:
                    raise ValueError(
                        f"Product {gr_node['op']} has invalid number of predecessors: ",
                        (len(preds), node),
                    )
            case "ie_map":
                if gr_node["name"] == "not":
                    if len(preds) != 1:
                        raise ValueError(
                            "Not node has invalid number of predecessors: ",
                            (len(preds), node),
                        )
            case "get_const" | "get_rel":
                return
            case _:
                raise ValueError(f"Unsupported operation: {gr_node['op']}")

    def prepare_node(
        self,
        graph: nx.DiGraph,
        node: str | int,
        nodes_schema_types_dict: dict[str | int, list[str]],
    ) -> None:
        """Prepares the node in the graph, calculating and updating the schema types dictionary."""
        gr_node = graph.nodes[node]
        preds = [
            {**graph.nodes[node_name], "node_name": node_name}
            for node_name in graph.pred[node]
        ]
        if node in nodes_schema_types_dict:
            return
        match gr_node["op"]:
            case "get_rel":
                nodes_schema_types_dict[node] = self.get_input_schema_types(node)
            case "rename" | "select" | "union":
                nodes_schema_types_dict[node] = nodes_schema_types_dict[
                    preds[0]["node_name"]
                ]
            case "project":
                pred = preds[0]
                nodes_schema_types_dict[node] = [
                    nodes_schema_types_dict[pred["node_name"]][
                        pred["schema"].index(col)
                    ]
                    for col in gr_node["schema"]
                ]
            case "product" | "join":
                schema_types = []
                for x in gr_node["schema"]:
                    try:
                        index = preds[0]["schema"].index(x)
                        schema_types.append(
                            nodes_schema_types_dict[preds[0]["node_name"]][index]
                        )
                    except ValueError:
                        index = preds[1]["schema"].index(x)
                        schema_types.append(
                            nodes_schema_types_dict[preds[1]["node_name"]][index]
                        )
                nodes_schema_types_dict[node] = schema_types
            case "groupby":
                schema_types = []
                for index, agg in enumerate(gr_node["agg"]):
                    if agg is None:
                        schema_types.append(
                            nodes_schema_types_dict[preds[0]["node_name"]][index]
                        )
                    elif (
                        agg == "count"
                    ):  # what about min / max should it match int if the field is int?
                        schema_types.append("DATA_TYPE_INT")
                    else:
                        schema_types.append("DATA_TYPE_FLOAT")
                nodes_schema_types_dict[node] = schema_types
            case "get_const":
                nodes_schema_types_dict[node] = [
                    self.PYTHON_TO_DATAFLOW_TYPES[type(const)]
                    for const in gr_node["const_dict"].values()
                ]
            case "ie_map":
                if callable(gr_node["in_schema"]):
                    gr_node["in_schema"] = gr_node["in_schema"](gr_node["in_arity"])
                if callable(gr_node["out_schema"]):
                    gr_node["out_schema"] = gr_node["out_schema"](gr_node["out_arity"])
                nodes_schema_types_dict[node] = [
                    (
                        self.PYTHON_TO_DATAFLOW_TYPES[t]
                        # TODO Support multiple types
                        if not isinstance(t, tuple)
                        else list(
                            filter(lambda x: x in self.PYTHON_TO_DATAFLOW_TYPES, t)
                        )[0]
                    )
                    for t in gr_node["in_schema"] + gr_node["out_schema"]
                ]
            case _:
                raise ValueError(f"Unsupported operation: {gr_node['op']}")

    def generate_node_code(
        self,
        graph: nx.DiGraph,
        node: str | int,
        nodes_schema_types_dict: dict[str | int, list[str]],
        anchor: str | int | None = None,
        in_iterate: bool = False,
    ) -> str:
        """Returns a string of rust code for the node in the graph."""
        gr_node = graph.nodes[node]
        self.validate_node(graph, node)
        self.prepare_node(graph, node, nodes_schema_types_dict)
        op_to_code_generator = {
            "get_rel": self.get_from_input_code,
            "rename": self.get_rename_code,
            "project": self.get_project_code,
            "join": self.get_join_code,
            "select": self.get_select_code,
            "union": self.get_union_code,
            "groupby": self.get_groupby_code,
            "get_const": self.get_from_input_code,
            "product": self.get_join_code,
            "ie_map": self.get_ie_map_code,
        }
        if gr_node["op"] not in op_to_code_generator:
            raise ValueError(f"Unsupported operation: {gr_node['op']}")
        return op_to_code_generator[gr_node["op"]](
            graph,
            node,
            anchor=anchor,
            in_iterate=in_iterate,
            nodes_schema_types_dict=nodes_schema_types_dict,
        )

    def get_ie_map_code(
        self,
        graph: nx.DiGraph,
        node: str | int,
        anchor: str | int | None = None,
        in_iterate: bool = False,
        nodes_schema_types_dict: dict[str | int, list[str]] = {},
    ) -> str:
        """Returns a string of rust code for ie_map node in the graph."""
        prev_nodes = list(graph.pred[node])
        node_str = self.get_node_str(node, anchor=anchor, in_iterate=in_iterate)
        prev_node_str = self.get_node_str(
            prev_nodes[0], anchor=anchor, in_iterate=in_iterate
        )
        # STD_IE_FUNCTIONS = {
        #     "rgx": {
        #         ("DATA_TYPE_STRING", "DATA_TYPE_STRING"): "rgx_str_span",
        #         ("DATA_TYPE_STRING", "DATA_TYPE_SPAN"): "rgx_span_span",
        #     },
        #     "as_str": {("DATA_TYPE_SPAN"): "span_as_str"},
        #     "deconstruct_span": {("DATA_TYPE_SPAN"): "deconstruct_span"},
        #     "rgx_is_match": {
        #         ("DATA_TYPE_STRING", "DATA_TYPE_STRING"): "rgx_is_match_str",
        #         ("DATA_TYPE_STRING", "DATA_TYPE_SPAN"): "rgx_is_match_span",
        #     },
        #     "span_contained": {("DATA_TYPE_SPAN", "DATA_TYPE_SPAN"): "span_contained"},
        #     "rgx_split": {
        #         ("DATA_TYPE_STRING", "DATA_TYPE_STRING"): "rgx_split_str",
        #         ("DATA_TYPE_STRING", "DATA_TYPE_SPAN"): "rgx_split_span",
        #     },
        # }

        gr_node = graph.nodes[node]
        in_schema = gr_node["schema"][: gr_node["in_arity"]]
        out_schema = gr_node["schema"][gr_node["in_arity"] :]
        match gr_node["name"]:
            case "rgx" as func_name:
                code = f"let {node_str} = {prev_node_str}.flat_map(|{self.get_col_schema(in_schema)}| {{ \n \
                    {func_name}({', '.join([f'&{col}' for col in in_schema])}).map(move |vec| ({', '.join([f'{col}.clone()' for col in in_schema]+[f'vec[{i}].clone()' for i in gr_node["out_arity"]])})) \n \
                }});"
            case (
                "as_str"
                | "rgx_split"
                | "rgx_is_match"
                | "deconstruct_span"
                | "span_contained" as func_name
            ):
                code = f"let {node_str} = {prev_node_str}.flat_map(|{self.get_col_schema(in_schema)}| {{ \n \
                    {func_name}({', '.join([f'&{col}' for col in in_schema])}).map(move |{self.get_col_schema(out_schema)}| ({', '.join([f'{col}.clone()' for col in in_schema]+out_schema)})) \n \
                }});"

            case "not":
                code = f"let {node_str} = {prev_node_str}.map(|{get_node_schema(graph, prev_nodes[0])}| ({get_node_schema(graph, prev_nodes[0])}, !{get_node_schema(graph, prev_nodes[0])}));"
            case _:
                ie_map_template = self._template_env.get_template("ie_map.rs.jinja2")
                code = ie_map_template.render(
                    output_node=node_str,
                    prev_node=prev_node_str,
                    grpc_address=f"http://{self._config.LISTEN_ADDRESS}",
                    function_name=graph.nodes[node]["name"],
                    in_schema=self.get_col_schema(
                        gr_node["schema"][: len(gr_node["in_schema"])]
                    ),
                    out_schema_len=len(gr_node["out_schema"]),
                    in_schema_len=len(gr_node["in_schema"]),
                    out_schema_types=nodes_schema_types_dict[node],
                    DATAFLOW_TO_RUST_TYPES=self.DATAFLOW_TO_RUST_TYPES,
                )
        return code

    def get_project_code(
        self,
        graph: nx.DiGraph,
        node: str | int,
        anchor: str | int | None = None,
        in_iterate: bool = False,
        nodes_schema_types_dict: dict[str | int, list[str]] | None = None,
    ) -> str:
        schema = get_node_schema(graph, node)
        prev_nodes = list(graph.pred[node])

        prev_node_str = self.get_node_str(
            prev_nodes[0], anchor=anchor, in_iterate=in_iterate
        )
        node_str = self.get_node_str(node, anchor=anchor, in_iterate=in_iterate)
        prev_schema = get_node_schema(graph, prev_nodes[0])
        return f"let {node_str} = {prev_node_str}.map(|{prev_schema}| {schema});"

    def get_from_input_code(
        self,
        graph: nx.DiGraph,
        node: str | int,
        anchor: str | int | None = None,
        in_iterate: bool = False,
        nodes_schema_types_dict: dict[str | int, list[str]] | None = None,
    ) -> str:
        node_str = self.get_node_str(node, anchor=anchor, in_iterate=in_iterate)
        return f"let {node_str} = input_{node}.to_collection(scope);"

    def get_rename_code(
        self,
        graph: nx.DiGraph,
        node: str | int,
        anchor: str | int | None = None,
        in_iterate: bool = False,
        nodes_schema_types_dict: dict[str | int, list[str]] | None = None,
    ) -> str:
        schema = self.get_col_schema(
            self.update_repeatable_cols_in_schema(graph.nodes[node]["schema"])
        )

        prev_nodes = list(graph.pred[node])

        prev_node_str = self.get_node_str(
            prev_nodes[0], anchor=anchor, in_iterate=in_iterate
        )
        node_str = self.get_node_str(node, anchor=anchor, in_iterate=in_iterate)

        return f"let {node_str} = {prev_node_str}.map(|{schema}| {schema});"

    def get_select_code(
        self,
        graph: nx.DiGraph,
        node: str | int,
        anchor: str | int | None = None,
        in_iterate: bool = False,
        nodes_schema_types_dict: dict[str | int, list[str]] | None = None,
    ) -> str:
        gr_node = graph.nodes[node]
        prev_nodes = list(graph.pred[node])

        prev_node_str = self.get_node_str(
            prev_nodes[0], anchor=anchor, in_iterate=in_iterate
        )
        node_str = self.get_node_str(node, anchor=anchor, in_iterate=in_iterate)

        theta = gr_node["theta"]
        predicates = []
        if hasattr(theta, "pos_val_tuples"):  #  equalConstTheta
            predicates = [f"*col_{pos} == {val}" for pos, val in theta.pos_val_tuples]
        elif hasattr(theta, "col_pos_tuples"):  #  equalColTheta
            predicates = [
                f"col_{pos1} == col_{pos2}" for pos1, pos2 in theta.col_pos_tuples
            ]
        else:
            raise ValueError(f"Unsupported theta join: {theta}. {dir(theta)}")
        return f"let {node_str} = {prev_node_str}.filter(|{self.get_col_schema([f"col_{i}" for i in range(len(gr_node['schema']))])}| {' && '.join(predicates)});"

    def update_repeatable_cols_in_schema(self, schema: list[str]) -> list[str]:
        repeatable_cols: dict[str, list[int]] = {}
        for i, col in enumerate(schema):
            if col in repeatable_cols:
                repeatable_cols[col].append(i)
            else:
                repeatable_cols[col] = [i]

        if not repeatable_cols:
            return schema
        result = schema.copy()
        for col, indecies in repeatable_cols.items():
            if len(indecies) > 1:
                for i, index in enumerate(indecies):
                    result[index] = f"{col}_{i}"
        return result

    def get_groupby_code(
        self,
        graph: nx.DiGraph,
        node: str | int,
        anchor: str | int | None = None,
        in_iterate: bool = False,
        nodes_schema_types_dict: dict[str | int, list[str]] | None = None,
    ) -> str:
        gr_node = graph.nodes[node]
        prev_nodes = list(graph.pred[node])

        prev_node_str = self.get_node_str(
            prev_nodes[0], anchor=anchor, in_iterate=in_iterate
        )
        node_str = self.get_node_str(node, anchor=anchor, in_iterate=in_iterate)

        agg = gr_node["agg"]
        schema = self.update_repeatable_cols_in_schema(gr_node["schema"])
        groupby_cols = [schema[i] for i, agg_func in enumerate(agg) if agg_func is None]
        agg_by_cols = {
            schema[i]: {"agg_func": agg_func}
            for i, agg_func in enumerate(agg)
            if agg_func is not None
        }

        agg_cols = list(agg_by_cols.keys())
        agg_by_index = {}

        for key, agg_func in agg_by_cols.items():
            agg_func["index"] = agg_cols.index(key)
            agg_by_index[agg_func["index"]] = {
                "agg_func": agg_func["agg_func"],
                "col_name": key,
            }

        multi_agg = False
        if len(agg_by_cols.values()) > 1:
            multi_agg = True

        declares = []
        agg_code = []
        output = []
        for col_name, agg_func in agg_by_cols.items():
            agg_index = agg_func["index"]
            agg_var = f"{agg_func['agg_func']}_{col_name}"
            output.append(agg_var)
            val = "val" if not multi_agg else f"val.{agg_index}"
            match agg_func["agg_func"]:
                case "sum":
                    declares.append(f"let mut {agg_var}: i32 = 0;")
                    agg_code.append(f"{agg_var} += {val} * (*cnt as i32);")
                case "count":
                    declares.append(f"let mut {agg_var}: i32 = 0;")
                    agg_code.append(f"{agg_var} += *cnt as i32;")
                case "max":
                    declares.append(f"let mut {agg_var}: i32 = i32::MIN;")
                    agg_code.append(f"{agg_var} = std::cmp::max({agg_var}, {val});")
                case "min":
                    declares.append(f"let mut {agg_var}: i32 = i32::MAX;")
                    agg_code.append(f"{agg_var} = std::cmp::min({agg_var}, {val});")
                case "avg":
                    declares.append(f"let mut {agg_var}: (i32, i32) = (0, 0);")
                    agg_code.append(f"{agg_var}.0 += {val} * (*cnt as i32);")
                    agg_code.append(f"{agg_var}.1 += *cnt as i32;")
                case _:
                    raise ValueError(
                        f"Unsupported aggregate function: {agg_func['agg_func']}"
                    )

        agg_template = self._template_env.get_template("aggregate.rs.jinja2")
        code = agg_template.render(
            output_node=node_str,
            prev_node=prev_node_str,
            in_schema=self.get_col_schema(schema),
            groupby_schema=self.get_col_schema(groupby_cols),
            groupby_len=len(groupby_cols),
            agg_len=len(agg_cols),
            agg_schema=self.get_col_schema(agg_cols),
            agg_decleration=declares,
            agg_list=agg_code,
            output_tuple=self.get_col_schema(output),
            agg_by_index=agg_by_index,
        )
        return code

    def generate_graph_code(
        self,
        graph: nx.DiGraph,
        nodes_schema_types_dict: dict[str | int, list[str]],
    ) -> list[str]:
        flow_code = list()
        iterate_template = self._template_env.get_template("iterate.rs.jinja2")
        reduced, cycles = reduced_graph(graph)
        for node in list(nx.topological_sort(reduced)):
            self.prepare_node(
                reduced, node, nodes_schema_types_dict=nodes_schema_types_dict
            )
            if node in cycles.keys():
                nodes_schema_types_dict[f"iter_{node}"] = nodes_schema_types_dict[node]
                self.prepare_node(
                    cycles[node],
                    f"iter_{node}",
                    nodes_schema_types_dict=nodes_schema_types_dict,
                )
                iter_graph = create_iter_graph(graph, cycles[node], node)
                anchor_code = self.generate_node_code(
                    reduced, node, nodes_schema_types_dict
                )
                cycle_code = {}
                cycle_order = traverse_cycle(cycles[node], f"iter_{node}")
                for cycle_node in cycle_order:
                    self.prepare_node(
                        iter_graph,
                        cycle_node,
                        nodes_schema_types_dict=nodes_schema_types_dict,
                    )
                    cycle_code[cycle_node] = self.generate_node_code(
                        iter_graph,
                        cycle_node,
                        nodes_schema_types_dict,
                        anchor=f"iter_{node}",
                        in_iterate=True,
                    )
                flow_code.append(
                    iterate_template.render(
                        {
                            "ingress_nodes": find_ingress_nodes(
                                graph, list(cycles[node].nodes)
                            ),
                            "anchor": node,
                            "cycle_flow": cycle_order,
                            "flow_code": cycle_code,
                            "anchor_code": anchor_code,
                        }
                    )
                )
            else:
                flow_code.append(
                    self.generate_node_code(reduced, node, nodes_schema_types_dict)
                )
        return flow_code

    def create_rust_file(self, timestamp: str, graph: nx.DiGraph) -> None:
        dest_path = self._config.GENERATED_RUST_PROJECT_PATH / f"{timestamp}.rs"
        template = self._template_env.get_template(self._config.RUST_FILE_TEMPLATE_NAME)
        nodes_schema_types_dict: dict[str | int, list[str]] = dict()
        flow_code = self.generate_graph_code(graph, nodes_schema_types_dict)
        output_node = find_output(graph)
        output_text = template.render(
            query_id=self._query_id,
            sources=self.get_sources_data(
                graph, nodes_schema_types_dict=nodes_schema_types_dict
            ),
            flow_code=flow_code,
            output_node_str=self.get_node_str(output_node),
            output_vars_count=len(graph.nodes[output_node]["schema"]),
        )
        self._config.RUST_SERVER_LIB_SRC_FILE_PATH.unlink(missing_ok=True)
        with open(dest_path, "w") as f:
            f.write(output_text)

        with open(self._config.RUST_SERVER_LIB_SRC_FILE_PATH, "w") as f:
            f.write(output_text)

        graph.graph["nodes_schema_types_dict"] = nodes_schema_types_dict

    def build_so(self, graph: nx.DiGraph) -> tuple[Path, str]:
        self._config.GENERATED_RUST_PROJECT_PATH.joinpath("src").mkdir(
            parents=True, exist_ok=True
        )
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self._query_id += 1
        self.create_rust_file(timestamp, graph)
        self.build_rust(
            self._config.RUST_SERVER_LIB_DIR_PATH.joinpath(
                self._cargo_file_name
            ).absolute(),
            self._config.RUST_SO_BUILD_LOG_PATH,
        )
        # Determine file extension based on the platform
        crate_name = "spannerflow"
        if os.name == "posix":  # Linux/macOS
            if os.uname().sysname == "Darwin":
                extension = ".dylib"  # macOS
            else:
                extension = ".so"  # Linux
            lib_filename = f"lib{crate_name}{extension}"
        elif os.name == "nt":  # Windows
            extension = ".dll"
            lib_filename = f"{crate_name}{extension}"
        else:
            raise RuntimeError("Unsupported OS")
        return (
            self._config.RUST_SERVER_LIB_DIR_PATH.joinpath(
                "target", "release", lib_filename
            ),
            f"query_{self._query_id}",
        )

    def build_rust(self, cargo_toml_path: Path, log_path: Path) -> None:
        command = [
            "cargo",
            "build",
            "--release",
            "--manifest-path",
            str(cargo_toml_path),
        ]

        self._config.LOGS_DIR.mkdir(parents=True, exist_ok=True)
        with open(log_path, "a") as log_file:
            subprocess.run(
                command,
                cwd=str(cargo_toml_path.parent),
                check=True,
                stderr=log_file,
                stdout=log_file,
            )

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()